# PDCP · N08 — E3 prompt / response-format ablation
**Notebook 8 of 12.** Kaggle: **GPU T4 x2** · Internet **On** · **Save & Run All (Commit)**. Expected ≈ 8–9 quota-hours (one session; resumable).

**Inputs:** `kbs-n3-output` (images, pairs, few-shot pool) · `kbs-n5-output` (E1 answers = P-Def baseline) · `kbs-n6-part2` (E2 mean-edge-length counterfactual images + P-Def answers on them).

**Key question.** E2 showed that shorter edges ("compactness") causally push models toward "lower stress". Does the **definition's wording** ("pairs few edges apart drawn close together") cause this? → compare the MEL effect under:
- **P-Def** (baseline, from E1/E2) · **P-NoDef** (no definition) · **P-Expert** (persona only) · **P-DefNeutral** (definition says explicitly that size/compactness/edge length do NOT matter).

**Other ablations** (base pairs only): **P-Struct** (checklist of NU/ELD/XR), **P-NoDiff** ("no clear difference" instead of "same"), **P-2AFC** (forced choice), **P-ReasonFirst**, **P-NoReason**, **P-FewShot3** (3 worked examples from held-out graphs).

**Design.** 200 base pairs (stratified half of the 400-subset, all Δstress × conditions), both image orders; for the 3 key prompts also their MEL counterfactuals (δ = −0.05 and +0.10), both orders.
Models: **InternVL-4B** and **Q3VL-2B** — all 9 variants; **Q25VL-7B** — key prompts + P-NoDiff + P-NoReason (slowest model).

**Outputs** (`/kaggle/working/n08_output/`): `results/e3/<model>.jsonl`, `e3_queries_<model>.csv`, `e3_summary.csv`, `e3_mel_effects.csv`, `fewshot.json`, `pdcp_infer.py` v0.4, figures, logs, `MANIFEST.txt`.

In [1]:
# ==== SETTINGS ====
N03_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n3-output"
N05_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n5-output"
N06_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n6-part2"
RESUME_ROOT = None
OUTPUT_DIR = "/kaggle/working/n08_output"
KEY_PROMPTS = ["P-NoDef", "P-Expert", "P-DefNeutral"]
OTHER_PROMPTS = ["P-Struct", "P-NoDiff", "P-2AFC", "P-ReasonFirst", "P-NoReason", "P-FewShot3"]
PLAN = {"InternVL-4B": KEY_PROMPTS + OTHER_PROMPTS,
        "Q25VL-7B": KEY_PROMPTS + ["P-NoDiff", "P-NoReason"],
        "Q3VL-2B": KEY_PROMPTS + OTHER_PROMPTS}
MODELS = ["Q25VL-7B", "InternVL-4B", "Q3VL-2B"]      # Q25VL alone on one GPU, the two fast models share the other
N_BASE = 200
MEL_DELTAS = [-0.05, 0.10]
SESSION_HOURS = 11.3
SEED = 2026

In [2]:
import os, sys, json, time, glob, shutil, subprocess, importlib
T0 = time.time()
for d in ["results/e3", "logs"]: os.makedirs(f"{OUTPUT_DIR}/{d}", exist_ok=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
import torch
N_GPU = torch.cuda.device_count(); assert N_GPU >= 1, "Set Accelerator = GPU T4 x2"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.57", "accelerate", "bitsandbytes", "qwen-vl-utils", "num2words"],
               capture_output=True, text=True)
import numpy as np, pandas as pd
def find(root, name):
    f = sorted(glob.glob(f"{root}/**/{name}", recursive=True)); assert f, f"{name} not found under {root}"; return os.path.dirname(f[0])
N03 = find(N03_ROOT, "pairs.parquet"); N06 = find(N06_ROOT, "mel_cf_pairs.parquet")
PR = pd.read_parquet(f"{N03}/pairs.parquet"); DR = pd.read_parquet(f"{N03}/drawings.parquet")
MCP = pd.read_parquet(f"{N06}/mel_cf_pairs.parquet")
png = dict(zip(DR.drawing_id, DR.png))
def path(did): return f"{N03}/{png[did]}" if did in png else f"{N06}/images/768/{did}.png"
print("N03:", N03, "| N06:", N06)

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

N03: /kaggle/input/datasets/resadussalehinrafsan/kbs-n3-output/n03_output | N06: /kaggle/input/datasets/resadussalehinrafsan/kbs-n6-part2/n06_output


## 1. Base pairs (200), MEL counterfactuals, few-shot examples

In [3]:
SUB = PR[(PR.set == "main") & PR.subset400].copy()
BASE = SUB.groupby(["dstress_design", "cond"], group_keys=False).sample(frac=min(1.0, N_BASE / len(SUB)), random_state=SEED)
if len(BASE) < N_BASE: BASE = pd.concat([BASE, SUB.drop(BASE.index).sample(min(N_BASE - len(BASE), len(SUB) - len(BASE)), random_state=SEED)])
BASE = BASE.head(N_BASE)
MEL = MCP[MCP.base_pair.isin(BASE.pair_id) & MCP.cf_delta.isin(MEL_DELTAS) & (MCP.cf_proxy == "mel")]
print(len(BASE), "base pairs |", len(MEL), "MEL counterfactual pairs |", BASE.dstress_design.value_counts().sort_index().to_dict())
# few-shot: 3 held-out pairs (answers: one '1', one '2', one 'same') with true stress values
FS = PR[PR.set == "fewshot"]; Di = DR.set_index("drawing_id")
FA = FS[FS.correct == "A"].sort_values("dstress_design", ascending=False)
pick = [FA.iloc[0], FA.iloc[min(1, len(FA) - 1)], (FS[FS.correct == "same"].iloc[0] if (FS.correct == "same").any() else FA.iloc[-1])]
fewshot = []
for k, r in enumerate(pick):
    a, b = path(r.drawing_A), path(r.drawing_B); ka, kb = Di.loc[r.drawing_A, "ksm"], Di.loc[r.drawing_B, "ksm"]
    if k == 1: a, b, ka, kb = b, a, kb, ka          # put the lower-stress drawing second once
    ans = "same" if r.correct == "same" else ("1" if ka < kb else "2")
    reason = (f"Both drawings have the same stress ({ka:.2f})." if ans == "same" else
              f"Image {ans} has lower stress ({min(ka, kb):.2f} vs {max(ka, kb):.2f}): its drawn distances match path distances more consistently.")
    fewshot.append({"img1": a, "img2": b, "answer_json": json.dumps({"answer": ans, "confidence": 5, "reason": reason})})
json.dump(fewshot, open(f"{OUTPUT_DIR}/fewshot.json", "w"), indent=1)
print([json.loads(f["answer_json"])["answer"] for f in fewshot])

200 base pairs | 384 MEL counterfactual pairs | {0.0: 39, 0.03: 36, 0.06: 41, 0.1: 36, 0.15: 35, 0.2: 13}
['1', '2', 'same']


## 2. Query lists per model (both orders, shuffled)

In [4]:
def rows_for(df, prompt, kind):
    out = []
    for r in df.itertuples():
        a, b = path(r.drawing_A), path(r.drawing_B)
        for order in ("AB", "BA"):
            img1, img2 = (a, b) if order == "AB" else (b, a)
            corr = "same" if r.correct == "same" else ("1" if (r.correct == "A") == (order == "AB") else "2")
            out.append({"pair_id": r.pair_id, "order": order, "prompt_id": prompt, "img1": img1, "img2": img2, "correct_img": corr,
                        "kind": kind, "base_pair": r.pair_id if kind == "base" else r.base_pair,
                        "cf_delta": getattr(r, "cf_delta", np.nan), "dstress": r.dstress_design, "cond": r.cond})
    return out
QQ = {}
for m, prompts in PLAN.items():
    rows = []
    for p in prompts:
        rows += rows_for(BASE, p, "base")
        if p in KEY_PROMPTS: rows += rows_for(MEL, p, "mel")
    Q = pd.DataFrame(rows).sample(frac=1, random_state=SEED)
    Q["_p"] = Q.prompt_id.map({p: i for i, p in enumerate(prompts)}); Q = Q.sort_values("_p", kind="stable").drop(columns="_p")
    Q.to_csv(f"{OUTPUT_DIR}/e3_queries_{m}.csv", index=False); QQ[m] = Q
    print(f"{m}: {len(Q)} queries", Q.groupby('prompt_id').size().to_dict())
assert all(os.path.exists(p) for Q in QQ.values() for p in set(Q.img1) | set(Q.img2)), "missing images"
if RESUME_ROOT:
    for f in glob.glob(f"{RESUME_ROOT}/**/results/e3/*.jsonl", recursive=True):
        shutil.copy(f, f"{OUTPUT_DIR}/results/e3/{os.path.basename(f)}"); print("resumed", os.path.basename(f))

InternVL-4B: 5904 queries {'P-2AFC': 400, 'P-DefNeutral': 1168, 'P-Expert': 1168, 'P-FewShot3': 400, 'P-NoDef': 1168, 'P-NoDiff': 400, 'P-NoReason': 400, 'P-ReasonFirst': 400, 'P-Struct': 400}
Q25VL-7B: 4304 queries {'P-DefNeutral': 1168, 'P-Expert': 1168, 'P-NoDef': 1168, 'P-NoDiff': 400, 'P-NoReason': 400}
Q3VL-2B: 5904 queries {'P-2AFC': 400, 'P-DefNeutral': 1168, 'P-Expert': 1168, 'P-FewShot3': 400, 'P-NoDef': 1168, 'P-NoDiff': 400, 'P-NoReason': 400, 'P-ReasonFirst': 400, 'P-Struct': 400}


## 3. Inference worker v0.4 (per-row prompt variants, few-shot)

In [5]:
%%writefile /kaggle/working/n08_output/pdcp_infer.py
"""
pdcp_infer.py — PDCP VLM inference worker (v0.4.0). One model per process.
v0.4: per-row prompt_id (many prompt variants per process), per-prompt max_new_tokens, few-shot examples (--fewshot).
v0.3: processor_kwargs API, per-model processor options + default batch, 25-word reasons, early abort on unparseable output.

Usage:
  python pdcp_infer.py --model Q3VL-4B --pairs pairs.csv --out results.jsonl --summary sum.json
                       [--prompt P-Def] [--batch 1] [--batch_test 4] [--max_new 160] [--hf_token TOKEN]

pairs.csv columns: pair_id, order (AB|BA), img1, img2 (absolute PNG paths), correct_img ("1"|"2"|"same")
Resumable: rows already in --out (same pair_id+order+prompt) are skipped.
"""
import argparse, gc, json, os, re, sys, time, traceback

MODELS = {   # batch = default batch size on one T4 (from the N04 pilot); proc = extra processor options
    "Q3VL-2B":     {"ids": ["Qwen/Qwen3-VL-2B-Instruct"], "quant": None, "batch": 8, "proc": {}},
    "Q3VL-4B":     {"ids": ["Qwen/Qwen3-VL-4B-Instruct"], "quant": None, "batch": 4, "proc": {}},
    "Q3VL-8B":     {"ids": ["Qwen/Qwen3-VL-8B-Instruct"], "quant": "nf4", "batch": 2, "proc": {}},
    "Q25VL-7B":    {"ids": ["Qwen/Qwen2.5-VL-7B-Instruct"], "quant": "nf4", "batch": 4, "proc": {}},
    "Gemma3-4B":   {"ids": ["google/gemma-3-4b-it"], "quant": None, "batch": 4, "proc": {}},
    "InternVL-4B": {"ids": ["OpenGVLab/InternVL3_5-4B-HF", "OpenGVLab/InternVL3-2B-hf"], "quant": None, "batch": 4,
                    "proc": {"crop_to_patches": False}},   # one 448-px tile per image (pilot: tiling -> 11 s/query, OOM at batch 4)
    "SmolVLM-2B":  {"ids": ["HuggingFaceTB/SmolVLM2-2.2B-Instruct"], "quant": None, "batch": 1, "proc": {}},
}

SYSTEM = ("You are a careful visual analyst. You will be shown two images of node-link diagrams that depict the SAME "
          "network (same nodes and same edges), drawn with different node positions. Answer the question strictly in "
          "the JSON format requested. Do not add any text outside the JSON.")
DEFINITION = ("In a node-link diagram, each circle is a node and each line is an edge. The \"stress\" of a drawing measures "
              "how well the straight-line distances between pairs of nodes match the graph-theoretic distances between them "
              "(the number of edges on the shortest path connecting them). A drawing has LOW stress when pairs of nodes that "
              "are few edges apart are drawn close together and pairs that are many edges apart are drawn far apart, "
              "consistently across the whole drawing. A drawing has HIGH stress when these distances disagree.")
JSON_SPEC = ('Respond with JSON only:\n{"answer": "1" | "2" | "same", "confidence": <integer 1-5>, '
             '"reason": "<at most 25 words>"}')
QUESTION = "\n\nImage 1 and Image 2 show the same network. Which image has LOWER stress?\n\n"
DEF_NEUTRAL = ("In a node-link diagram, each circle is a node and each line is an edge. The \"stress\" of a drawing measures how well "
               "the straight-line distances between ALL pairs of nodes are proportional to their graph-theoretic distances (the number of "
               "edges on the shortest path between them). Only this proportionality matters: the overall size of the drawing, how compact "
               "or spread out it looks, and the absolute length of the edges do not affect stress. A drawing has low stress when the ratio "
               "between drawn distance and path distance is the same for all pairs of nodes.")
STRUCT = ("\n\nEvaluate each image on three criteria, then decide:\n1. Node distribution: are nodes spread evenly, or clumped?\n"
          "2. Edge lengths: are edges of similar length, or do lengths vary a lot?\n3. Edge crossings: how many edges cross each other?\n"
          "Drawings with evenly spread nodes, uniform edge lengths and few crossings usually have lower stress, but judge the overall "
          "correspondence between distances and path lengths.")
J_STRUCT = ('Respond with JSON only:\n{"criteria": {"image1": {"node_distribution": "<short>", "edge_lengths": "<short>", "crossings": "<short>"}, '
            '"image2": {"node_distribution": "<short>", "edge_lengths": "<short>", "crossings": "<short>"}}, '
            '"answer": "1" | "2" | "same", "confidence": <integer 1-5>, "reason": "<at most 25 words>"}')
J_NODIFF = ('Respond with JSON only:\n{"answer": "1" | "2" | "no clear difference", "confidence": <integer 1-5>, "reason": "<at most 25 words>"}')
J_2AFC = ('You must choose one image. Respond with JSON only:\n{"answer": "1" | "2", "confidence": <integer 1-5>, "reason": "<at most 25 words>"}')
J_RFIRST = ('First think step by step in the "reason" field, then give the answer. Respond with JSON only:\n'
            '{"reason": "<at most 60 words>", "answer": "1" | "2" | "same", "confidence": <integer 1-5>}')
J_NOREASON = ('Respond with JSON only:\n{"answer": "1" | "2" | "same", "confidence": <integer 1-5>}')
PROMPTS = {
    "P-Def": DEFINITION + QUESTION + JSON_SPEC,
    "P-Expert": ("You are an expert in graph drawing and network visualization. Image 1 and Image 2 show the same network. "
                 "Which image has lower stress?\n\n" + JSON_SPEC),
    "P-NoDef": QUESTION.strip() + "\n\n" + JSON_SPEC,
    "P-DefNeutral": DEF_NEUTRAL + QUESTION + JSON_SPEC,
    "P-Struct": DEFINITION + STRUCT + QUESTION + J_STRUCT,
    "P-NoDiff": DEFINITION + QUESTION + J_NODIFF,
    "P-2AFC": DEFINITION + QUESTION + J_2AFC,
    "P-ReasonFirst": DEFINITION + QUESTION + J_RFIRST,
    "P-NoReason": DEFINITION + QUESTION + J_NOREASON,
    "P-FewShot3": DEFINITION + QUESTION + JSON_SPEC,     # + 3 worked examples inserted before the question
}
MAX_NEW = {"P-Struct": 230, "P-ReasonFirst": 150, "P-NoReason": 24}
FEWSHOT = []   # filled from --fewshot: list of {"img1","img2","answer_json"}


def parse_answer(text):
    out = {"answer": None, "confidence": None, "reason": None, "parse_ok": False}
    if not text:
        return out
    m = re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            obj = json.loads(m.group(0))
            a = str(obj.get("answer", "")).strip().lower().replace("image", "").strip()
            if a in ("1", "2"):
                out["answer"] = a
            elif a in ("same", "no clear difference"):
                out["answer"] = "same"
            c = obj.get("confidence")
            out["confidence"] = int(c) if c is not None and str(c).strip().isdigit() else None
            out["reason"] = obj.get("reason")
            out["parse_ok"] = out["answer"] is not None
            if out["parse_ok"]:
                return out
        except Exception:
            pass
    m = re.search(r'answer"?\s*(?:is)?\s*[:=]?\s*"?\s*(image\s*)?(1|2|same|no clear difference)\b', text, re.I)
    if m:
        a = m.group(2).lower()
        out["answer"] = "same" if a.startswith(("same", "no")) else a
        out["parse_ok"] = True
        r = re.search(r'"reason"\s*:\s*"([^"]*)', text)
        out["reason"] = r.group(1) if r else None
    return out


def build_messages(img1, img2, prompt_id, with_images=True):
    def im(x):
        return {"type": "image", "image": x} if with_images else {"type": "image"}
    content = [{"type": "text", "text": SYSTEM}]
    if prompt_id == "P-FewShot3" and FEWSHOT:
        content[0]["text"] += "\n\nFirst, three worked examples with the correct answers."
        for k, ex in enumerate(FEWSHOT, 1):
            content += [{"type": "text", "text": f"Example {k}. Image 1:"}, im(ex["img1_pil"]), {"type": "text", "text": "Image 2:"},
                        im(ex["img2_pil"]), {"type": "text", "text": "Correct answer: " + ex["answer_json"]}]
        content.append({"type": "text", "text": "Now the actual question."})
    content += [{"type": "text", "text": "Image 1:"}, im(img1), {"type": "text", "text": "Image 2:"}, im(img2),
                {"type": "text", "text": PROMPTS[prompt_id]}]
    return [{"role": "user", "content": content}]


class Runner:
    def __init__(self, key, hf_token=None, log=print):
        import torch
        from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
        self.torch, self.key, self.log = torch, key, log
        spec = MODELS[key]
        self.route, self.model, self.proc, self.model_id, self.dtype_name = None, None, None, None, None
        errors = []
        for mid in spec["ids"]:
            for dt in (["float16", "bfloat16"] if spec["quant"] is None else ["float16"]):
                try:
                    t0 = time.time()
                    kw = {"device_map": {"": 0}, "token": hf_token, "low_cpu_mem_usage": True}
                    if spec["quant"] == "nf4":
                        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                                        bnb_4bit_compute_dtype=torch.float16,
                                                                        bnb_4bit_use_double_quant=True)
                    else:
                        kw["dtype"] = getattr(torch, dt)
                    if self.model is None or self.model_id != mid:
                        self._free()
                        self.model = AutoModelForImageTextToText.from_pretrained(mid, **kw).eval()
                        self.proc = AutoProcessor.from_pretrained(mid, token=hf_token)
                    self.model_id, self.dtype_name, self.load_s = mid, (spec["quant"] or dt), time.time() - t0
                    return
                except Exception as e:
                    errors.append(f"{mid} {dt}: {type(e).__name__}: {str(e)[:300]}")
                    self.log("load failed: " + errors[-1])
                    self._free()
        raise RuntimeError("all loads failed | " + " || ".join(errors))

    def reload_dtype(self, dt):
        """Reload the same model in another dtype (used when fp16 produces garbage)."""
        from transformers import AutoModelForImageTextToText
        self._free(keep_proc=True)
        self.model = AutoModelForImageTextToText.from_pretrained(self.model_id, device_map={"": 0},
                                                                 dtype=getattr(self.torch, dt), low_cpu_mem_usage=True).eval()
        self.dtype_name = dt

    def _free(self, keep_proc=False):
        if self.model is not None:
            del self.model
            self.model = None
        if not keep_proc:
            self.proc = None
        gc.collect()
        try:
            self.torch.cuda.empty_cache()
        except Exception:
            pass

    def _inputs(self, batch, prompt_id):
        """batch = list of (img1, img2). Tries route A (chat template loads images) then B (processor call)."""
        torch = self.torch
        routes = [self.route] if self.route else ["A", "B_nested", "B_flat"]
        last = None
        for route in routes:
            try:
                if route == "A":
                    msgs = [build_messages(a, b, prompt_id) for a, b in batch]
                    if len(batch) > 1:
                        self.proc.tokenizer.padding_side = "left"
                    pk = dict(MODELS[self.key].get("proc", {}))
                    if len(batch) > 1:
                        pk["padding"] = True
                    inp = self.proc.apply_chat_template(msgs if len(batch) > 1 else msgs[0], add_generation_prompt=True,
                                                        tokenize=True, return_dict=True, return_tensors="pt",
                                                        processor_kwargs=pk)
                else:
                    texts = [self.proc.apply_chat_template(build_messages(a, b, prompt_id, with_images=False),
                                                           add_generation_prompt=True, tokenize=False) for a, b in batch]
                    ex = [x for e in FEWSHOT for x in (e["img1_pil"], e["img2_pil"])] if prompt_id == "P-FewShot3" else []
                    imgs = [ex + [a, b] for a, b in batch] if route == "B_nested" else [x for ab in batch for x in ex + list(ab)]
                    if len(batch) > 1:
                        self.proc.tokenizer.padding_side = "left"
                    inp = self.proc(text=texts, images=imgs, return_tensors="pt", padding=True, **MODELS[self.key].get("proc", {}))
                self.route = route
                dev = self.model.device
                mdt = torch.float16 if self.dtype_name in ("nf4", "float16") else getattr(torch, self.dtype_name)
                return {k: (v.to(dev, dtype=mdt) if torch.is_floating_point(v) else v.to(dev)) if hasattr(v, "to") else v
                        for k, v in inp.items()}
            except Exception as e:
                last = e
                if self.route:
                    raise
        raise RuntimeError(f"no input route worked: {type(last).__name__}: {last}")

    def generate(self, batch, prompt_id, max_new=160):
        torch = self.torch
        inp = self._inputs(batch, prompt_id)
        n_in = inp["input_ids"].shape[1]
        with torch.inference_mode():
            out = self.model.generate(**inp, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None, top_k=None)
        texts = self.proc.batch_decode(out[:, n_in:], skip_special_tokens=True)
        return texts, int(n_in)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True); ap.add_argument("--pairs", required=True)
    ap.add_argument("--out", required=True); ap.add_argument("--summary", required=True)
    ap.add_argument("--prompt", default="P-Def"); ap.add_argument("--batch", type=int, default=0, help="0 = model default")
    ap.add_argument("--min_parse", type=float, default=0.5, help="abort if parse rate after 40 queries is below this")
    ap.add_argument("--deadline", type=float, default=0, help="unix time after which the worker stops cleanly")
    ap.add_argument("--batch_test", type=int, default=0); ap.add_argument("--max_new", type=int, default=160)
    ap.add_argument("--hf_token", default=None); ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--fewshot", default=None, help="json list of {img1,img2,answer_json}")
    a = ap.parse_args()
    import pandas as pd, torch
    from PIL import Image
    global FEWSHOT
    def log(s): print(f"[{a.model} {time.strftime('%H:%M:%S')}] {s}", flush=True)
    summ = {"model": a.model, "prompt": a.prompt, "ok": False, "error": None}
    t_start = time.time()
    try:
        P = pd.read_csv(a.pairs)
        if a.limit: P = P.head(a.limit)
        done = set()
        if os.path.exists(a.out):
            for line in open(a.out):
                try:
                    r = json.loads(line); done.add((r["pair_id"], r["order"], r["prompt_id"]))
                except Exception:
                    pass
        if "prompt_id" not in P.columns: P["prompt_id"] = a.prompt
        todo = [r for r in P.to_dict("records") if (r["pair_id"], r["order"], r["prompt_id"]) not in done]
        order_p = {p: i for i, p in enumerate(dict.fromkeys(P.prompt_id))}
        todo.sort(key=lambda r: order_p[r["prompt_id"]])            # stable: keeps the shuffle inside each prompt
        if a.fewshot:
            for e in json.load(open(a.fewshot)):
                e["img1_pil"] = Image.open(e["img1"]).convert("RGB"); e["img2_pil"] = Image.open(e["img2"]).convert("RGB"); FEWSHOT.append(e)
            log(f"loaded {len(FEWSHOT)} few-shot examples")
        log(f"{len(P)} pairs, {len(todo)} to do")
        torch.cuda.reset_peak_memory_stats()
        R = Runner(a.model, a.hf_token, log)
        log(f"loaded {R.model_id} as {R.dtype_name} in {R.load_s:.0f}s")
        summ.update({"model_id": R.model_id, "dtype": R.dtype_name, "load_s": round(R.load_s, 1)})
        load = lambda p: Image.open(p).convert("RGB")

        # warm-up + dtype sanity: if fp16 gives zero parseable answers on 3 items, retry in bf16
        if todo:
            w = todo[:3]
            texts, _ = R.generate([(load(r["img1"]), load(r["img2"])) for r in w[:1]], w[0]["prompt_id"], MAX_NEW.get(w[0]["prompt_id"], a.max_new))
            ok = parse_answer(texts[0])["parse_ok"]
            if not ok:
                t2 = [R.generate([(load(r["img1"]), load(r["img2"]))], r["prompt_id"], MAX_NEW.get(r["prompt_id"], a.max_new))[0][0] for r in w[1:]]
                if not any(parse_answer(t)["parse_ok"] for t in t2) and R.dtype_name == "float16":
                    log("fp16 output unparseable on warm-up -> retrying in bfloat16. sample: " + repr(texts[0][:200]))
                    try:
                        R.reload_dtype("bfloat16"); summ["dtype"] = "bfloat16"
                    except Exception as e:
                        log(f"bf16 reload failed ({type(e).__name__}); reloading fp16 and continuing")
                        R.reload_dtype("float16"); summ["dtype_note"] = "fp16 unparseable, bf16 unavailable"
        summ["route"] = R.route

        f = open(a.out, "a")
        times, n_ok, n_corr, n = [], 0, 0, 0
        bs = a.batch if a.batch > 0 else MODELS[a.model].get("batch", 1)
        summ["batch"] = bs
        groups, cur = [], []
        for r in todo:                       # chunks never mix prompt variants
            if cur and (r["prompt_id"] != cur[0]["prompt_id"] or len(cur) == bs):
                groups.append(cur); cur = []
            cur.append(r)
        if cur: groups.append(cur)
        for chunk in groups:
            pid = chunk[0]["prompt_id"]; mn = MAX_NEW.get(pid, a.max_new)
            if a.deadline and time.time() > a.deadline:
                log("deadline reached - stopping cleanly (resumable)"); summ["stopped_at_deadline"] = True; break
            t0 = time.time()
            try:
                texts, n_in = R.generate([(load(r["img1"]), load(r["img2"])) for r in chunk], pid, mn)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if bs > 1:
                    log("OOM at batch %d -> falling back to batch 1" % bs); bs = 1
                    texts, n_in = [], 0
                    for r in chunk:
                        t, n_in = R.generate([(load(r["img1"]), load(r["img2"]))], pid, mn); texts += t
                else:
                    raise
            dt = (time.time() - t0) / len(chunk)
            for r, t in zip(chunk, texts):
                p = parse_answer(t)
                corr = p["answer"] == str(r["correct_img"]) if p["parse_ok"] else None
                rec = {"pair_id": r["pair_id"], "order": r["order"], "model": a.model, "model_id": R.model_id,
                       "dtype": R.dtype_name, "prompt_id": pid, "raw_text": t, "parsed": p, "parse_ok": p["parse_ok"],
                       "correct_img": str(r["correct_img"]), "correct": corr, "latency_s": round(dt, 3), "input_tokens": n_in,
                       "batch": len(chunk), "ts": time.strftime("%Y-%m-%dT%H:%M:%S")}
                f.write(json.dumps(rec) + "\n")
                n += 1; n_ok += p["parse_ok"]; n_corr += bool(corr)
            f.flush(); times.append(dt)
            if n >= 40 and n - len(chunk) < 40 and n_ok / n < a.min_parse:
                log(f"parse rate {n_ok/n:.2f} < {a.min_parse} after {n} queries - aborting this model")
                summ["aborted_low_parse"] = True; break
            if n % 50 < len(chunk):
                log(f"{n}/{len(todo)} | {dt:.2f}s/q | parse {n_ok/n:.2f} | acc {n_corr/max(1,n_ok):.2f}")
        f.close()
        summ.update({"n": n, "sec_per_query": round(float(sum(times) / max(1, len(times))), 3), "parse_rate": round(n_ok / max(1, n), 3),
                     "acc_on_parsed": round(n_corr / max(1, n_ok), 3), "input_tokens": n_in if todo else None})

        # optional batched-throughput test on the first items (not written to results)
        if a.batch_test and a.batch_test > 1 and len(P) >= a.batch_test * 3:
            rows = P.to_dict("records")[: a.batch_test * 3]
            try:
                t0 = time.time()
                for i in range(0, len(rows), a.batch_test):
                    texts, _ = R.generate([(load(r["img1"]), load(r["img2"])) for r in rows[i:i + a.batch_test]], a.prompt, a.max_new)
                summ["batch_test"] = {"batch": a.batch_test, "sec_per_query": round((time.time() - t0) / len(rows), 3),
                                      "parse_rate": round(sum(parse_answer(t)["parse_ok"] for t in texts) / len(texts), 3)}
            except Exception as e:
                summ["batch_test"] = {"batch": a.batch_test, "error": f"{type(e).__name__}: {str(e)[:200]}"}
            log(f"batch test: {summ['batch_test']}")
        summ["vram_peak_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)
        summ["ok"] = True
    except Exception as e:
        summ["error"] = f"{type(e).__name__}: {str(e)[:1500]}"
        log("FAILED: " + summ["error"]); traceback.print_exc()
    summ["wall_s"] = round(time.time() - t_start, 1)
    json.dump(summ, open(a.summary, "w"), indent=2)
    log(f"done: {json.dumps({k: summ.get(k) for k in ['ok', 'n', 'sec_per_query', 'parse_rate', 'acc_on_parsed', 'vram_peak_gb']})}")


if __name__ == "__main__":
    main()


Writing /kaggle/working/n08_output/pdcp_infer.py


## 4. Run

In [6]:
HF_CACHE = "/tmp/hf_cache"
sys.path.insert(0, OUTPUT_DIR); import pdcp_infer as I; importlib.reload(I)
DEADLINE = T0 + SESSION_HOURS * 3600
def ndone(m):
    f = f"{OUTPUT_DIR}/results/e3/{m}.jsonl"; return sum(1 for _ in open(f)) if os.path.exists(f) else 0
queue = [m for m in MODELS if ndone(m) < len(QQ[m])]; running = {}
def launch(model, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), HF_HUB_CACHE=HF_CACHE, TOKENIZERS_PARALLELISM="false",
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    cmd = [sys.executable, f"{OUTPUT_DIR}/pdcp_infer.py", "--model", model, "--pairs", f"{OUTPUT_DIR}/e3_queries_{model}.csv",
           "--out", f"{OUTPUT_DIR}/results/e3/{model}.jsonl", "--summary", f"{OUTPUT_DIR}/results/e3/{model}.summary.json",
           "--max_new", "96", "--deadline", str(DEADLINE), "--fewshot", f"{OUTPUT_DIR}/fewshot.json", "--min_parse", "0.3"]
    logf = open(f"{OUTPUT_DIR}/logs/{model}.log", "a")
    running[gpu] = (model, subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=env), logf)
    print(f"[{(time.time()-T0)/60:6.1f} min] start {model} on GPU {gpu}")
last = 0
while queue or running:
    for gpu in range(N_GPU):
        if gpu not in running and queue and time.time() < DEADLINE - 1800: launch(queue.pop(0), gpu)
    if not running: break
    time.sleep(30)
    if time.time() - last > 1800:
        last = time.time(); print("   " + " | ".join(f"{m}: {ndone(m)}/{len(QQ[m])}" for m in MODELS))
    for gpu, (model, p, logf) in list(running.items()):
        if p.poll() is not None:
            logf.close(); del running[gpu]
            print(f"[{(time.time()-T0)/60:6.1f} min] end {model}: {open(f'{OUTPUT_DIR}/logs/{model}.log').read().strip().splitlines()[-1][-250:]}")
            for mid in I.MODELS[model]["ids"]: shutil.rmtree(f"{HF_CACHE}/models--{mid.replace('/', '--')}", ignore_errors=True)
print(f"run phase finished after {(time.time()-T0)/3600:.2f} h")

[   0.7 min] start Q25VL-7B on GPU 0
[   0.7 min] start InternVL-4B on GPU 1
   Q25VL-7B: 0/4304 | InternVL-4B: 0/5904 | Q3VL-2B: 0/5904
   Q25VL-7B: 272/4304 | InternVL-4B: 948/5904 | Q3VL-2B: 0/5904
   Q25VL-7B: 588/4304 | InternVL-4B: 2084/5904 | Q3VL-2B: 0/5904
   Q25VL-7B: 900/4304 | InternVL-4B: 3116/5904 | Q3VL-2B: 0/5904
   Q25VL-7B: 1216/4304 | InternVL-4B: 3752/5904 | Q3VL-2B: 0/5904
   Q25VL-7B: 1544/4304 | InternVL-4B: 4520/5904 | Q3VL-2B: 0/5904
   Q25VL-7B: 1872/4304 | InternVL-4B: 5004/5904 | Q3VL-2B: 0/5904
[ 198.7 min] end InternVL-4B: [InternVL-4B 23:54:37] done: {"ok": false, "n": null, "sec_per_query": null, "parse_rate": null, "acc_on_parsed": null, "vram_peak_gb": null}
[ 198.7 min] start Q3VL-2B on GPU 1
   Q25VL-7B: 2200/4304 | InternVL-4B: 5508/5904 | Q3VL-2B: 344/5904
   Q25VL-7B: 2508/4304 | InternVL-4B: 5508/5904 | Q3VL-2B: 1224/5904
   Q25VL-7B: 2808/4304 | InternVL-4B: 5508/5904 | Q3VL-2B: 2088/5904
   Q25VL-7B: 3116/4304 | InternVL-4B: 5508/5904 | Q3VL-2B

## 5. Results: per-prompt behaviour and the MEL (compactness) effect by prompt

In [7]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
def load(root, sub, prompt=None):
    R = []
    for f in glob.glob(f"{root}/**/results/{sub}/*.jsonl", recursive=True): R += [json.loads(l) for l in open(f)]
    R = pd.DataFrame(R)
    if not len(R): return R
    if prompt: R["prompt_id"] = R.get("prompt_id", prompt)
    a = R.parsed.map(lambda p: p.get("answer"))
    R["choice"] = np.where(a == "same", "same", np.where(a.isin(["1", "2"]), np.where((a == "1") == (R.order == "AB"), "A", "B"), None))
    R["ans"] = a
    return R
E3 = load(OUTPUT_DIR, "e3")
E1 = load(N05_ROOT, "e1", "P-Def"); E2 = load(N06_ROOT, "e2", "P-Def")
base_ids, mel_ids = set(BASE.pair_id), set(MEL.pair_id)
ref = pd.concat([E1[E1.pair_id.isin(base_ids)], E2[E2.pair_id.isin(mel_ids)]]) if len(E1) else pd.DataFrame()
ref = ref[ref.model.isin(PLAN)] if len(ref) else ref
ALL = pd.concat([E3, ref], ignore_index=True)
meta = pd.concat([BASE.assign(kind="base", base_pair=BASE.pair_id, cf_delta=np.nan), MEL.assign(kind="mel")])[
    ["pair_id", "kind", "base_pair", "cf_delta", "dstress_design", "cond"]]
meta["key"] = meta.pair_id.map(dict(zip(pd.concat([BASE, MEL]).pair_id, pd.concat([BASE, MEL]).correct)))
ALL = ALL.merge(meta, on="pair_id", how="inner")
ALL["parse_ok"] = ALL.choice.notna()
ALL["correct"] = (ALL.choice == ALL.key)          # recomputed from the answer and the true lower-stress drawing
status = {m: {"done": ndone(m), "total": len(QQ[m])} for m in MODELS}
status["all_complete"] = all(v["done"] >= v["total"] for v in status.values())
json.dump(status, open(f"{OUTPUT_DIR}/e3_status.json", "w"), indent=2); print(status)

rows = []
for (m, p), g in ALL[ALL.kind == "base"].groupby(["model", "prompt_id"]):
    ok = g[g.parse_ok]; nz = ok[ok.dstress_design > 0]; z = ok[ok.dstress_design == 0]
    pA = lambda d: (d.choice == "A").mean()
    rows.append({"model": m, "prompt": p, "n": len(g), "parse": round(g.parse_ok.mean(), 3),
                 "acc_dstress>0": round(nz.correct.mean(), 3), "p_same_all": round((ok.ans == "same").mean(), 3),
                 "p_same_when_equal": round((z.ans == "same").mean(), 3), "p_image1": round((ok.ans == "1").mean(), 3),
                 "congruent_minus_conflict": round(nz[nz.cond == "congruent"].correct.mean() - nz[nz.cond.str.startswith("conflict")].correct.mean(), 3),
                 "lure_gap_dstress0": round(pA(z[z.cond == "congruent"]) - pA(z[z.cond == "conflict_all"]), 3),
                 "mean_conf": round(pd.to_numeric(ok.parsed.map(lambda q: q.get("confidence")), errors="coerce").mean(), 2)})
SUM = pd.DataFrame(rows); SUM.to_csv(f"{OUTPUT_DIR}/e3_summary.csv", index=False)
pd.set_option("display.width", 220); print(SUM.to_string(index=False))

# MEL effect: shift in order-averaged P(choose A) from base pair to its MEL counterfactual, within the same prompt
pa = ALL[ALL.choice.isin(["A", "B"])].assign(a=lambda d: (d.choice == "A").astype(float)).groupby(["model", "prompt_id", "pair_id"]).a.mean().reset_index()
pa = pa.merge(meta[["pair_id", "kind", "base_pair", "cf_delta"]], on="pair_id")
b = pa[pa.kind == "base"][["model", "prompt_id", "base_pair", "a"]].rename(columns={"a": "a_base"})
x = pa[pa.kind == "mel"].merge(b, on=["model", "prompt_id", "base_pair"])
x["shift"] = x.a - x.a_base
EFF = x.groupby(["model", "prompt_id", "cf_delta"]).agg(n=("shift", "size"), shift=("shift", "mean"),
                                                        se=("shift", lambda s: s.std() / np.sqrt(len(s)))).reset_index()
EFF["z"] = EFF["shift"] / EFF.se
# compactness sensitivity = shift(shorter) - shift(longer): larger = stronger "compact = low stress" bias
cs = EFF.pivot_table(index=["model", "prompt_id"], columns="cf_delta", values="shift")
cs["compactness_bias"] = cs[-0.05] - cs[0.10]
EFF.round(4).to_csv(f"{OUTPUT_DIR}/e3_mel_effects.csv", index=False)
print(cs.round(3).to_string())

fig, ax = plt.subplots(1, 2, figsize=(15, 4.3))
order = ["P-Def"] + KEY_PROMPTS
cb = cs.compactness_bias.unstack(0).reindex(order)
cb.plot.bar(ax=ax[0]); ax[0].axhline(0, c="k", lw=.8); ax[0].set_title("Compactness bias = shift(MEL −0.05) − shift(MEL +0.10)"); ax[0].set_xlabel("")
acc = SUM.pivot(index="prompt", columns="model", values="acc_dstress>0").reindex(["P-Def"] + KEY_PROMPTS + OTHER_PROMPTS)
acc.plot.bar(ax=ax[1]); ax[1].axhline(0.5, ls=":", c="grey"); ax[1].set_ylim(0, 1); ax[1].set_title("Accuracy (Δstress>0, both orders) by prompt"); ax[1].set_xlabel("")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_e3_prompts.png", dpi=120); plt.show()
print("ALL COMPLETE" if status["all_complete"] else "NOT COMPLETE -> save output as dataset, set RESUME_ROOT, run again")

{'Q25VL-7B': {'done': 4304, 'total': 4304}, 'InternVL-4B': {'done': 5508, 'total': 5904}, 'Q3VL-2B': {'done': 5512, 'total': 5904}, 'all_complete': False}
      model        prompt   n  parse  acc_dstress>0  p_same_all  p_same_when_equal  p_image1  congruent_minus_conflict  lure_gap_dstress0  mean_conf
InternVL-4B        P-2AFC 400  1.000          0.559       0.000              0.000     0.365                     0.218              0.375       4.03
InternVL-4B         P-Def 400  1.000          0.565       0.000              0.000     0.560                     0.269              0.542       4.00
InternVL-4B  P-DefNeutral 400  1.000          0.621       0.000              0.000     0.510                     0.218              0.458       4.56
InternVL-4B      P-Expert 400  1.000          0.599       0.000              0.000     0.112                    -0.036              0.000       4.91
InternVL-4B    P-FewShot3   4  1.000          0.500       0.000                NaN     1.000        

In [8]:
shutil.rmtree(f"{OUTPUT_DIR}/__pycache__", ignore_errors=True)
lines = [f"N08 E3 · {(time.time()-T0)/3600:.2f} h · complete: {status['all_complete']} · " + " | ".join(f"{m}: {status[m]['done']}/{status[m]['total']}" for m in MODELS), ""]
for root, _, files in os.walk(OUTPUT_DIR):
    for f in sorted(files):
        p = os.path.join(root, f); lines.append(f"{os.path.relpath(p, OUTPUT_DIR):40s} {os.path.getsize(p)/1024:9.1f} KB")
open(f"{OUTPUT_DIR}/MANIFEST.txt", "w").write("\n".join(lines)); print("\n".join(lines))

N08 E3 · 6.90 h · complete: False · Q25VL-7B: 4304/4304 | InternVL-4B: 5508/5904 | Q3VL-2B: 5512/5904

e3_mel_effects.csv                             1.2 KB
e3_queries_InternVL-4B.csv                  1445.3 KB
e3_queries_Q25VL-7B.csv                     1064.8 KB
e3_queries_Q3VL-2B.csv                      1445.3 KB
e3_status.json                                 0.2 KB
e3_summary.csv                                 1.7 KB
fewshot.json                                   1.1 KB
fig_e3_prompts.png                            57.3 KB
pdcp_infer.py                                 20.8 KB
results/e3/InternVL-4B.jsonl                3759.6 KB
results/e3/InternVL-4B.summary.json            0.8 KB
results/e3/Q25VL-7B.jsonl                   2455.6 KB
results/e3/Q25VL-7B.summary.json               0.3 KB
results/e3/Q3VL-2B.jsonl                    5031.5 KB
results/e3/Q3VL-2B.summary.json                0.8 KB
logs/InternVL-4B.log                          19.0 KB
logs/Q25VL-7B.log                